# Word2Vec: Skip-gram with Negative Sampling (SGNS)

**The two core coding tasks**

1. Implement a dataset that creates sliding-window positive pairs and simple uniform negative samples.
2. Implement the SGNS neural network and loss with two independent embedding tables.

The text download, cleaning, vocabulary, training loop, and embedding tests are provided. Frequency-based negative sampling and CBOW are optional stretch tasks.

## 0. Setup

In [ ]:
import math
import os
import random
import re
import time
import urllib.request
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

EMBEDDING_DIM = 50
WINDOW_SIZE = 3
NUM_NEG_SAMPLES = 5
MIN_COUNT = 5
EPOCHS = 10
BATCH_SIZE = 1024
MAX_TOKENS = 80_000
LEARNING_RATE = 3e-3
DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("CPU threads:", torch.get_num_threads())

## 1. Data preprocessing

In [ ]:
def download_book(timeout=20):
    headers = {"User-Agent": "Mozilla/5.0 (educational SGNS notebook)"}

    try:
        request = urllib.request.Request('https://www.gutenberg.org/cache/epub/1661/pg1661.txt', headers=headers)
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8", errors="replace")
            print(f"Downloaded {len(raw):,} characters from Project Gutenberg.")
            return raw, "Project Gutenberg: The Adventures of Sherlock Holmes"
    except Exception as exc:
        print(f"Download attempt failed ({type(exc).__name__})")


def strip_gutenberg_boilerplate(text):
    """Keep the book body when Gutenberg start/end markers are available."""
    start_match = re.search(r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.I)
    end_match = re.search(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.I)
    start = start_match.end() if start_match else 0
    end = end_match.start() if end_match else len(text)
    return text[start:end]


def tokenize(text):
    """Lowercase and return word tokens while removing punctuation and numbers."""
    text = text.lower().replace("’", "'")
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text)


raw_text, corpus_source = download_book()
tokens = tokenize(strip_gutenberg_boilerplate(raw_text))[:MAX_TOKENS]
print(f"Source: {corpus_source}")
print(f"Tokens retained: {len(tokens):,}")
print("First 20 tokens:", tokens[:20])

### Build the vocabulary

Filtering first ensures every retained token has an index. Sorting by decreasing frequency makes results reproducible; alphabetical order breaks ties.


In [ ]:
raw_counts = Counter(tokens)
kept_words = sorted(
    (word for word, count in raw_counts.items() if count >= MIN_COUNT),
    key=lambda word: (-raw_counts[word], word),
)

word2idx = {word: idx for idx, word in enumerate(kept_words)}
idx2word = {idx: word for word, idx in word2idx.items()}
filtered_tokens = [word for word in tokens if word in word2idx]
token_ids = [word2idx[word] for word in filtered_tokens]
vocab_counts = torch.tensor(
    [raw_counts[idx2word[idx]] for idx in range(len(idx2word))],
    dtype=torch.float32,
)

print(f"Vocabulary size: {len(word2idx):,}")
print(f"Tokens after rare-word filtering: {len(token_ids):,}")
print("Ten most frequent words:", [(w, raw_counts[w]) for w in kept_words[:10]])

assert len(word2idx) == len(idx2word) == len(vocab_counts)
assert all(0 <= token_id < len(word2idx) for token_id in token_ids)


## Coding Task 1: Dataset

**Expected output from `__getitem__`**

A tuple `(target, positive_context, negative_contexts)`. The standard DataLoader will combine these into `[B]`, `[B]`, and `[B, K]` batches.

Hints:
- 1 target with $k$-window gives at most $2k$ positive samples.
- Use `max` and `min` to handle windows near the start and end of the corpus.
- Build a `set` of word IDs in the current window; repeatedly call `randrange(vocab_size)` until enough valid negatives are collected.

In [ ]:
class SkipGramDataset(Dataset):
    def __init__(self, token_ids, vocab_size, window_size, num_neg_samples, seed=42):
        self.token_ids = torch.as_tensor(token_ids, dtype=torch.long)
        self.vocab_size = int(vocab_size)
        self.window_size = int(window_size)
        self.num_neg_samples = int(num_neg_samples)
        self.rng = random.Random(seed)

        #
        # ### Your code here ###
        #

        # Hint: create a list of target and positive indices

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):

        #
        # ### Your code here ###
        #

        return target, positive_context, negative_contexts

### Prepare dataloader + tests

In [ ]:
dataset = SkipGramDataset(
    token_ids=token_ids,
    vocab_size=len(word2idx),
    window_size=WINDOW_SIZE,
    num_neg_samples=NUM_NEG_SAMPLES,
    seed=SEED,
)

loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
    drop_last=False,
)

sample_targets, sample_positives, sample_negatives = next(iter(train_loader))
print(f"Positive training pairs: {len(dataset):,}")
print("Target batch shape:  ", tuple(sample_targets.shape))
print("Positive batch shape:", tuple(sample_positives.shape))
print("Negative batch shape:", tuple(sample_negatives.shape))

assert sample_targets.dtype == torch.long
assert sample_positives.dtype == torch.long
assert sample_negatives.dtype == torch.long
assert sample_negatives.shape == (len(sample_targets), NUM_NEG_SAMPLES)

print("\nFirst five decoded examples:")
for i in range(5):
    target, positive, negatives = dataset[i]
    print(
        idx2word[target.item()], "->", idx2word[positive.item()],
        "| negatives:", [idx2word[j.item()] for j in negatives],
    )

## Coding Task 2: the SGNS model

Word2Vec learns **two independent tables**:

- target/input vectors $v_w$;
- context/output vectors $u_w$.

For a positive pair $(w,c)$ and sampled negatives $n_1,\ldots,n_K$, the loss for one example is

$$-\log \sigma(v_w^\top u_c)-\sum_{k=1}^{K}\log\sigma(-v_w^\top u_{n_k}).$$

**Expected inputs to `forward`**

- `targets`: word indices with shape `[B]`;
- `positive_contexts`: word indices with shape `[B]`;
- `negative_contexts`: word indices with shape `[B, K]`.

**Expected output**

One scalar tensor: the mean SGNS loss for the batch. Calling `.backward()` on it must create gradients for both embedding tables.

**Hints**

1. Create two separate `nn.Embedding(vocab_size, embedding_dim)` modules: one for targets and one for contexts. Never share their weights.
2. Looking up `[B]` indices produces `[B, D]`; looking up `[B, K]` indices produces `[B, K, D]`.
3. Compute positive dot products with element-wise multiplication followed by `sum(dim=1)`.
4. For negatives, use `torch.bmm`: multiply `[B, K, D]` by target vectors reshaped to `[B, D, 1]`.
5. Use `torch.nn.functional.logsigmoid`. Positive scores enter as-is; negative scores enter with a minus sign. Sum over the `K` negatives, combine both terms, negate, and average over the batch.

In [ ]:
class SkipGramNegativeSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        #
        # ### Your code here ###
        #

        # Model

        # Weights initialization

    def forward(self, targets, positive_contexts, negative_contexts):
        """
            targets:           [B]
            positive_contexts: [B]
            negative_contexts: [B, K]
        """

        #
        # ### Your code here ###
        #

        # positive loss

        # negative loss

        return loss

### Tests

In [ ]:
model = SkipGramNegativeSampling(len(word2idx), EMBEDDING_DIM).to(DEVICE)

test_loss = model(
    sample_targets[:8].to(DEVICE),
    sample_positives[:8].to(DEVICE),
    sample_negatives[:8].to(DEVICE),
)
assert test_loss.ndim == 0 and torch.isfinite(test_loss)
test_loss.backward()
model.zero_grad(set_to_none=True)
print(f"Smoke test passed; initial loss = {test_loss.item():.4f}")

## 3. Training loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = []
training_start = time.perf_counter()

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    running_loss = 0.0
    examples_seen = 0

    for targets, positives, negatives in train_loader:
        targets = targets.to(DEVICE)
        positives = positives.to(DEVICE)
        negatives = negatives.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        loss = model(targets, positives, negatives)
        loss.backward()
        optimizer.step()

        batch_examples = targets.size(0)
        running_loss += loss.item() * batch_examples
        examples_seen += batch_examples

    mean_loss = running_loss / examples_seen
    history.append(mean_loss)
    elapsed = time.perf_counter() - epoch_start
    print(f"Epoch {epoch:>2}/{EPOCHS} | mean loss {mean_loss:.4f} | {elapsed:.1f}s")

total_elapsed = time.perf_counter() - training_start
print(f"Training complete in {total_elapsed:.1f}s")


### Extract the final word embeddings

Hint: retrieve the input/target nn.Embedding layer from your model, then access its .weight tensor. Detach it from the computation graph, move it to the CPU, and clone it.

In [ ]:
#
# ### Your code here ###
#

final_embeddings = # ...

normalized_embeddings = F.normalize(final_embeddings, p=2, dim=1)

assert final_embeddings.shape == (len(word2idx), EMBEDDING_DIM)
print("Final embedding matrix:", tuple(final_embeddings.shape))

## 4. Semantic arithmetic evaluation

SGNS is stochastic, and this deliberately small single-book corpus is not expected to reproduce large-benchmark Word2Vec results. Treat these as qualitative inspection tools. Neighbors should often share topic or syntactic role; analogies are a much stricter test and may vary.


In [ ]:
def _require_words(*words):
    missing = [word for word in words if word not in word2idx]
    if missing:
        raise KeyError(
            f"Out-of-vocabulary word(s): {missing}. "
            "Try a frequent word from the displayed vocabulary."
        )


def get_closest_words(word, top_n=5):
    """Print and return the nearest target embeddings by cosine similarity."""
    _require_words(word)
    if top_n < 1:
        raise ValueError("top_n must be at least 1")

    query_idx = word2idx[word]
    query_vector = final_embeddings[query_idx].unsqueeze(0)
    similarities = F.cosine_similarity(query_vector, final_embeddings, dim=1)
    similarities[query_idx] = -torch.inf

    k = min(top_n, len(word2idx) - 1)
    scores, indices = torch.topk(similarities, k=k)
    results = [(idx2word[idx.item()], score.item()) for score, idx in zip(scores, indices)]

    print(f"Closest words to '{word}':")
    for neighbor, score in results:
        print(f"  {neighbor:<16} cosine={score:.3f}")
    return results

def solve_analogy(word_a, word_b, word_c, top_n=5):
    """Solve a:b :: c:? using target-embedding vector arithmetic."""
    _require_words(word_a, word_b, word_c)
    if top_n < 1:
        raise ValueError("top_n must be at least 1")

    a, b, c = (word2idx[word] for word in (word_a, word_b, word_c))
    result_vector = final_embeddings[b] - final_embeddings[a] + final_embeddings[c]
    similarities = F.cosine_similarity(result_vector.unsqueeze(0), final_embeddings, dim=1)
    similarities[[a, b, c]] = -torch.inf

    k = min(top_n, len(word2idx) - 3)
    scores, indices = torch.topk(similarities, k=k)
    results = [(idx2word[idx.item()], score.item()) for score, idx in zip(scores, indices)]

    print(f"{word_b} - {word_a} + {word_c} -> {results[0][0]} (cosine={results[0][1]:.3f})")
    return results[0][0] if top_n == 1 else results

### Check 1: nearest neighbors


In [ ]:
probe_words = [
    "holmes",
    "watson",
    "london",
    "detective",
    "doctor",
    "police",
    "crime",
    "murder",
]

for probe in probe_words:
    if probe in word2idx:
        get_closest_words(probe, top_n=5)
        print()

### Check 2: word analogy solver

For `solve_analogy(a, b, c)`, we compute $v_b-v_a+v_c$ and return its closest vocabulary vector. The three input words are excluded from the answer.


In [ ]:
solve_analogy("detective", "holmes", "doctor", top_n=5)
solve_analogy("england", "london", "france", top_n=5)
solve_analogy("man", "husband", "woman", top_n=5)
solve_analogy("man", "king", "woman", top_n=5)

## Stretch tasks

### 1. Frequency-based negative sampling

Replace uniform random negatives with the original Word2Vec distribution

$$P(w_i)=\frac{f(w_i)^{0.75}}{\sum_j f(w_j)^{0.75}}.$$

The provided `vocab_counts` tensor contains frequencies in vocabulary-index order. Compute the probabilities once in `__init__`, then use `torch.multinomial(..., replacement=True)` to draw candidates. Continue rejecting words from the positive window. Compare runtime and nearest neighbors with uniform sampling.

### 2. CBOW

Implement Continuous Bag of Words using the same vocabulary and corpus. CBOW reverses the prediction direction: average the embeddings of all context words in the window and use that average to predict the center word against one positive output and several negative outputs. Your model should accept context indices shaped `[B, C]` (plus a mask if windows have different lengths) and return one scalar negative-sampling loss. Compare CBOW and Skip-gram on rare versus frequent probe words.

## Reflection questions

1. Why are the target and context embedding tables separate during training?
2. Why must obvious positive-window words be rejected as negatives?
3. What does uniform negative sampling emphasize differently from frequency-based sampling?
4. Why are stable analogies harder to learn from one short book than useful nearest neighbors?
5. Which experiment changed quality most, and what evidence from the fixed probes supports that conclusion?
